# R²IE Quickstart

An experimental architecture combining a VQ bottleneck (Mass Compressor), adaptive per-token compute (ACT Transformation Field), and bounded fast-weight adaptation (Condensation Loop).

> Status: experimental research prototype. Not benchmarked against production models.


In [ ]:
import torch
from r2ie.config import ModelConfig
from r2ie.data import CharDataset, CharTokenizer, load_corpus_text
from r2ie.model import R2IEModel
from torch.utils.data import DataLoader

text = load_corpus_text('fixture')
tok = CharTokenizer(text)
ds = CharDataset(text, seq_len=32, tokenizer=tok)
loader = DataLoader(ds, batch_size=16, shuffle=True, drop_last=True)
cfg = ModelConfig(vocab_size=tok.vocab_size, d_model=64, n_heads=2, n_layers=2, d_ff=128, codebook_size=32, max_ponder_steps=3)
model = R2IEModel(cfg)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
import torch.nn.functional as F
model.train()
it = iter(loader)
for step in range(1, 201):
    try:
        x, y = next(it)
    except StopIteration:
        it = iter(loader); x, y = next(it)
    opt.zero_grad()
    logits, aux = model(x)
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
    (ce + aux['commitment_loss'] + aux['ponder_cost']).backward()
    opt.step()
    if step % 40 == 0:
        print(step, round(ce.item(), 3))

In [ ]:
# Generate from a prompt using the fast-weight (Condensation Loop) modulation.
from r2ie.generate import build_parser, main as generate_main
model.eval()
# Or use the CLI: python -m r2ie.generate --checkpoint checkpoints/r2ie_latest.pt --prompt 'THE '